# Topology contribution

The first table compares MeSH-only with raw concatenation; the second compares
input-level fusion methods with latent gating. All methods use one optimizer
update per pair on the same 25 folds.

The saved tables can be viewed without rerunning the models.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../../data/README.md) and
[DataPreparation.ipynb](../../../data/DataPreparation.ipynb), you can rerun the
ablation and regenerate the tables using the functions in
[`src/dtpkg/ablation/`](../../../src/dtpkg/ablation/).

## Paths

Fill in `PROJECT_ROOT` to read the saved tables. The remaining placeholders are
needed for training; `REFERENCE_DIR` is the completed [transductive run](../../04_transductive_fusion/README.md).

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = ...  # DTP-KG checkout
DATA_DIR = ...  # Prepared private data
GRAPH_PATH = ...  # Canonical GraphML network
REFERENCE_DIR = ...  # Completed transductive run from experiment 04

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

EXPERIMENT = PROJECT_ROOT / "experiments" / "06_ablation" / "topology_contribution"
TABLES = EXPERIMENT / "tables"
RESULTS_ROOT = EXPERIMENT / "results" / "runs"
OUTPUT_DIR = EXPERIMENT / "results" / "table_exports"
RUN_TRAINING = False

## Table 1 — Topology contribution

In [ ]:
from IPython.display import display
from dtpkg.ablation.topology_reporting import load_topology_tables

report = load_topology_tables(TABLES)
display(report["paper_table"])

,Model,AUROC mean ± SD,ΔAUROC (model − MeSH only),Holm p (AUROC),F1 mean ± SD,ΔF1 (model − MeSH only),Holm p (F1)
0,MeSH only,0.9599 ± 0.0063,—,—,0.8955 ± 0.0132,—,—
1,Raw concat,0.9744 ± 0.0049,+0.01458**,0.0040,0.9210 ± 0.0107,+0.02543**,0.0050


Values are fold means ± sample SD. Holm adjustment retains all six tests,
including the fixed and learned weighting controls. Paired uncertainty uses
approximate corrected-CV intervals.

## Table 2 — Fusion methods

Differences are method minus latent gate. These six comparisons form a separate
Holm family. Raw methods combine input features; latent gating uses separate encoders.

In [ ]:
display(report["fusion_paper_table"])

,Model,AUROC mean ± SD,ΔAUROC (model − Latent gate),Holm p (AUROC),F1 mean ± SD,ΔF1 (model − Latent gate),Holm p (F1)
0,Raw concat,0.9744 ± 0.0049,-0.01401**,0.0003,0.9210 ± 0.0107,-0.03187**,0.0003
1,Fixed α=4,0.9774 ± 0.0050,-0.01111**,0.0011,0.9252 ± 0.0104,-0.02765**,0.0011
2,Learned α,0.9767 ± 0.0041,-0.01180**,0.0003,0.9257 ± 0.0087,-0.02710**,0.0003
3,Latent gate,0.9885 ± 0.0033,—,—,0.9528 ± 0.0079,—,—


## Regenerate the results

Complete the [transductive experiment](../../04_transductive_fusion/TransductiveFusion.ipynb)
and set `REFERENCE_DIR` to its run directory, then enable `RUN_TRAINING`.
The sequence below recreates the models and both tables.

In [ ]:
if RUN_TRAINING:
    from dtpkg.ablation.workflow import prepare_study
    from dtpkg.ablation.input_fusion import prepare_input_fusion, run_input_fusion
    from dtpkg.ablation.raw_weighted_fusion import prepare_raw_weighted, run_raw_weighted
    from dtpkg.ablation.topology_reporting import export_topology_tables

    fusion = prepare_study(
        study="fusion_methods",
        data_dir=DATA_DIR,
        graph_path=GRAPH_PATH,
        reference_dir=REFERENCE_DIR,
        results_root=RESULTS_ROOT,
    )
    fusion.run()
    input_run = RESULTS_ROOT / "input_fusion"
    weighted_run = RESULTS_ROOT / "raw_weighted_fusion"
    prepare_input_fusion(fusion.output_dir, input_run)
    run_input_fusion(input_run)
    prepare_raw_weighted(input_run, weighted_run)
    run_raw_weighted(weighted_run)
    export_topology_tables(input_run, weighted_run, OUTPUT_DIR)
    new_report = load_topology_tables(OUTPUT_DIR)
    display(new_report["paper_table"])
    display(new_report["fusion_paper_table"])